Basic RAG Limitations:
1) Single query perspective (misses relevant docs) -> A more advanced system can generate multiple versions of your question
2) No metadata filetring (retrieves irrelevant content) -> Metadata filtering = telling the retriever where it is allowed to search.
3) Full chunks returned (Noise in context) -> The LLM has to figure out Which part of this huge chunk actually answers the question.Advanced RAG can retrieve or extract only the relevant portion instead of Full chunks = giving the LLM more information than it actually needs.
4) Keyword OR semantic (not both together) -> Keyword search(to catch exact important terms) AND Semantic search(to catch meaning/similar wording) This is called hybrid search.

Advanced RAG Solutions:
1) Multi-Query Retriever -> Multiple perspectives
2) Self-Query Retriever -> Auto metadata filters
3) Contextual Compression ->Extract relevant parts
4) Hybrid Search -> Keyword(BM25 algorithm) + semantic -> ensemble retriever weighted combination
5) Parent document retriever (to reduce noise of full chunks) -> a document split into large parent chunks, then  split that particular parent into smaller child chunks.So the retriever searches the small child chunks.Small chunks are very specific and precise but can lack context, The parent contains the surrounding information. The parent chunk is returned to the LLM.(Small chunks for precise matching, large chunks for full context -? best of both accurate+complete)

(You search using the sentence, because it's precise.But once you find it, you give the LLM the page/paragraph, because it contains context.)

MUTI QUERY RETRIEVER : 
original query -> LLM generates variations -> each query searches -> results merged and de-duplicated(merged unique docs)
SELF QUERY RETRIEVER:
semantic search + metadata (auto generated) can understand the filters within the query(question)

In [ ]:
# Advanced RAG Patterns
# Multi-query, self-query, Contextual compression, hybrid search



from langchain_classic.retrievers import multi_query
from langchain_classic.retrievers import document_compressors
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import logging

# Enable logging to see multi-query-generation
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

#Sample Knowledge Base for demos
TECH_DOCS=[
     Document(
        page_content="""
        Python is a high-level, interpreted programming language created by
        Guido van Rossum. It was first released in 1991. Python emphasizes
        readability and simplicity and is widely used in data science,
        machine learning, web development, automation, and scripting.
        """,
        metadata={
            "language": "Python",
            "creator": "Guido van Rossum",
            "year_created": 1991,
            "paradigm": ["object-oriented", "procedural", "functional"],
            "type": "interpreted",
            "difficulty": "beginner",
            "use_cases": [
                "data science",
                "machine learning",
                "web development",
                "automation"
            ],
            "company": "Python Software Foundation",
            "extension": ".py",
            "category": "programming_language",
            "popularity": "very_high",
            "source": "programming_languages_guide.pdf",
            "page": 1,
            "country_origin": "Netherlands"
        }
    ),

    Document(
        page_content="""
        JavaScript is a high-level programming language primarily used to
        create interactive web pages. It was created by Brendan Eich in 1995.
        JavaScript runs in web browsers and can also be used on servers through
        environments such as Node.js. It is one of the core technologies of
        the modern web.
        """,
        metadata={
            "language": "JavaScript",
            "creator": "Brendan Eich",
            "year_created": 1995,
            "paradigm": ["object-oriented", "functional", "event-driven"],
            "type": "interpreted",
            "difficulty": "beginner",
            "use_cases": [
                "web development",
                "frontend",
                "backend",
                "mobile development"
            ],
            "runtime": ["Browser", "Node.js", "Deno"],
            "extension": ".js",
            "category": "programming_language",
            "popularity": "very_high",
            "source": "programming_languages_guide.pdf",
            "page": 2,
            "country_origin": "United States"
        }
    ),

    Document(
        page_content="""
        Java is a general-purpose, object-oriented programming language
        created by James Gosling and his team at Sun Microsystems. It was
        officially released in 1995. Java follows the principle of
        write once, run anywhere through the Java Virtual Machine (JVM).
        It is widely used for enterprise applications, backend systems,
        and Android development.
        """,
        metadata={
            "language": "Java",
            "creator": "James Gosling",
            "year_created": 1995,
            "paradigm": ["object-oriented", "imperative"],
            "type": "compiled_to_bytecode",
            "difficulty": "intermediate",
            "use_cases": [
                "enterprise applications",
                "backend",
                "Android",
                "large-scale systems"
            ],
            "runtime": "JVM",
            "extension": ".java",
            "category": "programming_language",
            "popularity": "very_high",
            "source": "programming_languages_guide.pdf",
            "page": 3,
            "country_origin": "Canada"
        }
    ),

    Document(
        page_content="""
        C++ is a general-purpose programming language developed by Bjarne
        Stroustrup as an extension of the C programming language. Development
        began in 1979 and the language was originally called C with Classes.
        C++ provides low-level memory control and is widely used in systems
        programming, game development, embedded systems, and competitive
        programming.
        """,
        metadata={
            "language": "C++",
            "creator": "Bjarne Stroustrup",
            "year_created": 1985,
            "paradigm": [
                "object-oriented",
                "procedural",
                "generic",
                "functional"
            ],
            "type": "compiled",
            "difficulty": "advanced",
            "use_cases": [
                "systems programming",
                "game development",
                "embedded systems",
                "competitive programming"
            ],
            "extension": ".cpp",
            "category": "programming_language",
            "popularity": "very_high",
            "source": "programming_languages_guide.pdf",
            "page": 4,
            "country_origin": "Denmark"
        }
    ),

    Document(
        page_content="""
        Go, also known as Golang, is a statically typed compiled programming
        language developed at Google by Robert Griesemer, Rob Pike, and Ken
        Thompson. Go was designed for simplicity, concurrency, and efficient
        development of large-scale software systems. It is commonly used for
        cloud services, APIs, networking, and distributed systems.
        """,
        metadata={
            "language": "Go",
            "creator": [
                "Robert Griesemer",
                "Rob Pike",
                "Ken Thompson"
            ],
            "year_created": 2009,
            "paradigm": ["procedural", "concurrent"],
            "type": "compiled",
            "difficulty": "intermediate",
            "use_cases": [
                "cloud computing",
                "APIs",
                "networking",
                "distributed systems",
                "DevOps"
            ],
            "company": "Google",
            "extension": ".go",
            "category": "programming_language",
            "popularity": "high",
            "source": "programming_languages_guide.pdf",
            "page": 5,
            "country_origin": "United States"
        }
    ),

    Document(
        page_content="""
        Rust is a systems programming language focused on performance,
        memory safety, and concurrency. It was originally developed by
        Graydon Hoare and later sponsored by Mozilla. Rust provides memory
        safety without requiring a garbage collector and is used for systems
        software, networking, WebAssembly, and performance-critical
        applications.
        """,
        metadata={
            "language": "Rust",
            "creator": "Graydon Hoare",
            "year_created": 2010,
            "paradigm": [
                "systems",
                "functional",
                "imperative"
            ],
            "type": "compiled",
            "difficulty": "advanced",
            "use_cases": [
                "systems programming",
                "WebAssembly",
                "networking",
                "embedded systems"
            ],
            "sponsor": "Mozilla",
            "extension": ".rs",
            "category": "programming_language",
            "popularity": "high",
            "source": "programming_languages_guide.pdf",
            "page": 6,
            "country_origin": "United States"
        }
    ),

    Document(
        page_content="""
        C is a general-purpose procedural programming language created by
        Dennis Ritchie at Bell Labs. It was developed in the early 1970s and
        became influential in operating systems and systems programming.
        The C language provides direct memory manipulation and is known for
        its performance and portability.
        """,
        metadata={
            "language": "C",
            "creator": "Dennis Ritchie",
            "year_created": 1972,
            "paradigm": ["procedural", "imperative"],
            "type": "compiled",
            "difficulty": "intermediate",
            "use_cases": [
                "operating systems",
                "embedded systems",
                "systems programming",
                "compilers"
            ],
            "extension": ".c",
            "category": "programming_language",
            "popularity": "very_high",
            "source": "programming_languages_guide.pdf",
            "page": 7,
            "country_origin": "United States"
        }
    ),

    Document(
        page_content="""
        TypeScript is a programming language developed by Microsoft that
        extends JavaScript by adding static typing and other language
        features. TypeScript code is compiled into JavaScript and can run
        wherever JavaScript runs. It is widely used for large-scale frontend
        and backend applications.
        """,
        metadata={
            "language": "TypeScript",
            "creator": "Anders Hejlsberg",
            "year_created": 2012,
            "paradigm": [
                "object-oriented",
                "functional",
                "generic"
            ],
            "type": "transpiled",
            "difficulty": "intermediate",
            "use_cases": [
                "frontend",
                "backend",
                "large-scale applications",
                "web development"
            ],
            "parent_language": "JavaScript",
            "company": "Microsoft",
            "extension": ".ts",
            "category": "programming_language",
            "popularity": "very_high",
            "source": "programming_languages_guide.pdf",
            "page": 8,
            "country_origin": "United States"
        }
    )
]


def create_base_vectorstore():
    "Create a basic vector store for demos."""
    return Chroma.from_documents (
    documents=TECH_DOCS,
    embedding=OpenAIEmbeddings (model="text-embedding-3-small")
    )
def demo_multi_query_retriever():
    "Multi-Query Retriever generates multiple query perspectives."""
    print("=" * 60)
    print("MULTI-QUERY RETRIEVER")
    print("Generates multiple perspectives on your question")
    print("=" * 60)
    vectorstore = create_base_vectorstore()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    # Create multi-query retriever
    retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever (search_kwargs={"k": 2}), llm=llm
    ) #llm=llm argument passes the language model necessary to rewrite those queries. 

    query = "What tools can I use to build AI applications?"
    print(f"\nOriginal Query: {query}")
    print("\nThe retriever will generate multiple query variations...")
    print("(Check INFO logs above for generated queries)\n")
    # Retrieve documents
    docs = retriever.invoke(query)

    print(f"Retrieved {len (docs)} unique documents:")

    for i, doc in enumerate(docs):
        print(
            f"\n{i+1}. [{doc.metadata.get('topic', 'N/A')}] {doc.page_content[:100]}"
        )

def demo_contextual_compression():
    """Contextual Compression extracts only relevant parts."""
    print("=" * 60)
    print("CONTEXTUAL COMPRESSION RETRIEVER")
    print("Extracts only query-relevant content from documents")
    print("=" * 60)

    vectorstore = create_base_vectorstore()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    # Create compressor
    compressor = LLMChainExtractor.from_llm(llm)
    #configure the retriever
    compression_retriever = ContextualCompressionRetriever(
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    compressor=compressor,
    )
    
    query="What frameworks exist for building LLM applications?"

    print(f"\nQuery: {query}")

    # Without compression
    base_docs = vectorstore.as_retriever (search_kwargs={"k": 2}).invoke(query)
    print(f"\n--- WITHOUT Compression (full chunks) ---")

    for doc in base_docs:
        print(f"Length: {len(doc.page_content)} chars")
        print(f"Content:{doc.page_content[:150]}...\n")
    
    # With compression
    compressed_docs = compression_retriever.invoke(query) #Look at a retrieved document + the question and extract only the relevant part.
    print(f"\n-- WITH Compression (relevant only) ---")

    for doc in compressed_docs:
        print(f"Length: {len (doc.page_content)} chars")
        print(f"Content: {doc.page_content}\n")


def demo_ensemble_hybrid_search():
    print("=" * 60)
    print("ENSEMBLE/HYBRID RETRIEVER")
    print("Combines keyword (BM25) + semantic search")
    print("=" * 60)

    vectorstore = create_base_vectorstore()

    #BM25 keyword retriever
    bm25_retriever = BM25Retriever.from_documents(TECH_DOCS)
    bm25_retriever.k= 3

    #Semantic retriever
    semantic_retriever=vectorstore.as_retriever(search_kwargs={"k":3})

    #Setting up an ensemble retriever: Retriever that ensembles multiple retrievers. It uses a rank fusion.
    #each retriever has its own ranking system.
    #Run multiple retrievers, collect their results, and combine/rank those results into one final list.
    #A fusion algorithm gives documents scores based on their positions in the different result lists.
    #One common approach is Reciprocal Rank Fusion (RRF). Higher rank->Higher fusion score->Higher final position
    #Use multiple retrievers and fuse their rankings to produce one better-ranked result list.

    ensemble_retriever=EnsembleRetriever(
        retrievers=[bm25_retriever,semantic_retriever,],
        weights=[0.4, 0.6] #40% keyword,60% semantic
    )

    # Test queries
    queries = [
    "PostgreSQL pgvector", #Keyword-heavy (BM25 helps)
    "What database stores embeddings?", # Semantic (vectors help)
    ]

    for query in queries:
    
        print(f"\nQuery: {query}")
        print("-" * 40)
        
        # Compare results
        bm25_results = bm25_retriever.invoke(query)
        semantic_results = semantic_retriever.invoke(query)
        ensemble_results = ensemble_retriever.invoke(query)
        print(f"BM25 top resat: rest: result: {bm25_results[0].page_content[:60]}...")
        print(f"Semantic top result: {semantic_results[0].page_content[:60]}...")
        print(f"Ensemble top result: {ensemble_results[0].page_content[:60]}...")

def demo_parent_document_retriever():
    """Parent Document Retriever: small chunks for search, large for context."""
    print("=" * 60)
    print("PARENT DOCUMENT RETRIEVER")
    print("Small chunks for precise search, large chunks for context")
    print("=" * 60)
        
        
if __name__ == "__main__":
    demo_multi_query_retriever()


"What tools can I use to build AI applications?"
                    ↓
                  LLM
                    ↓
       Multiple query variations
          ↓        ↓        ↓
         Q1       Q2       Q3
          ↓        ↓        ↓
        Vector   Vector   Vector
         DB       DB       DB
          ↓        ↓        ↓
        Docs     Docs     Docs
          └────────┼────────┘
                   ↓
            Combine + deduplicate
                   ↓
             List[Document]

Parent Document Retriever in LangChain is used when you want:

Small chunks for accurate retrieval
But larger parent documents/chunks for the LLM's context

The idea is:

Search using child chunks → retrieve their parent chunks/documents → send the parent content to the LLM.

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever #Go into this package/module and bring this particular class/function into my code
from langchain_core.stores import InMemoryStore #to temporarily store the parent documents
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

#InMemoryStore is essentially a temporary storage mechanism that can store objects in memory using keys.
#It lives in your application's memory.If your Python program stops, the data is generally gone.
# 1. Create vector store
vectorstore = Chroma(
    collection_name="child_documents",
    embedding_function=OpenAIEmbeddings()
)

# 2. Store for parent documents
store = InMemoryStore()

# 3. Parent splitter
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

# 4. Child splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

# 5. Create Parent Document Retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 6. Add documents
docs = [
    Document(
        page_content="""
        LangChain is a framework for building applications using
        large language models. It provides tools for retrieval,
        agents, memory, prompts, document processing and much more.
        """
    )
]

retriever.add_documents(docs)

#Query
query="What is LangGraph used for?"

print(f"\nQuery: {query}")

# Regular retrieval (would get small chunks)
child_docs = vectorstore.similarity_search(query, k = 1 )
print(f"\n--- Child Chunk (what search found) --")
print(f"Length: {len (child_docs [0].page_content)} chars")
print(f"Content: {child_docs[0].page_content}")


# Parent retrieval (gets full context)
parent_docs = retriever.invoke(query)
print(f"\n--- Parent Chunk (what's returned) ")
print(f"Length: {len (parent_docs[0].page_content)} chars")
print("Content preview: {parent_docs[0].page_content[:300]}..."

                Parent Document Retriever
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
        Vector Store             Docstore
          (Chroma)           (InMemoryStore)
             ↓                       ↓
        CHILD chunks             PARENTS
             ↓                       ↓
       Used for search        Used for context

                 QUERY
                   ↓
          ParentDocumentRetriever
                   ↓
          Search CHILDREN
                   ↓
             Chroma
                   ↓
          Relevant Child
                   ↓
          Find Parent ID
                   ↓
          InMemoryStore
                   ↓
            Parent Document
                   ↓
              parent_docs

#COMBINING MULTI QUERY AND COMPRESSION STRATEGIES

In [ ]:
def demo_advanced_rag_chain():

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    multi_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever (search_kwargs={"k": 2}), llm=llm
    ) 

    # Compression to focus on relevant info
    compressor = LLMChainExtractor.from_llm(llm)

    advanced_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=multi_retriever
    )

    prompt=ChatPromptTemplate.from_template("""
    Answer the following questions based on the following context, be specific and cite which technology will be used.
    Context: {context}
    Question:{question}
    Answer:""")

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = (
    {
        "context": advanced_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
    )

    # Test
    questions = [
    "What options do I have for building AI agents?",
    "How can I store and search embeddings?",
    ]

    for q in questions:
        print(f"\nQ: {q}")
        answer = rag_chain.invoke(q)
        print(f"A: {answer}")
        


    